In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
import gc
import datetime

warnings.filterwarnings("ignore")

In [ ]:
products = ["ind_ahor_fin_ult1", "ind_aval_fin_ult1", "ind_cco_fin_ult1",
            "ind_cder_fin_ult1", "ind_cno_fin_ult1",  "ind_ctju_fin_ult1", 
            "ind_ctma_fin_ult1", "ind_ctop_fin_ult1", "ind_ctpp_fin_ult1",
            "ind_deco_fin_ult1", "ind_deme_fin_ult1", "ind_dela_fin_ult1", 
            "ind_ecue_fin_ult1", "ind_fond_fin_ult1", "ind_hip_fin_ult1",
            "ind_plan_fin_ult1", "ind_pres_fin_ult1", "ind_reca_fin_ult1", 
            "ind_tjcr_fin_ult1", "ind_valo_fin_ult1", "ind_viv_fin_ult1",
            "ind_nomina_ult1",   "ind_nom_pens_ult1", "ind_recibo_ult1"]

In [ ]:
#Dados históricos de 4 meses para teste e 4 meses para treino

#Usando dados de 4 meses iniciais de 2015 para treino 
train_data_dfs = [pd.read_csv(f"data/train_2015_0{i}_28.csv") for i in range(1, 6)]

#Usando dados de 4 meses iniciais de 2016 para teste
test_data_dfs = [pd.read_csv(f"data/train_2016_0{i}_28.csv") for i in range(1, 5)]
test_data_dfs.append(pd.read_csv("data/test_dataset_new.csv"))

#Lista de produtos financeiros adicionados nesse mês
added_products = pd.read_csv('data/added_product_2015_04_28.csv')

In [ ]:
#Populando o arquivo de teste "test_dataset_new" com valores 0 nos produtos
for col in products:
    test_data_dfs[-1][col] = [0] * test_data_dfs[-1].shape[0]

In [ ]:
combined = pd.concat([pd.concat(train_data_dfs), pd.concat(test_data_dfs)]).reset_index(drop=True)
del train_data_dfs, test_data_dfs

In [ ]:
def process_data_from_original_dataframe(combined, encoders=None):
    #fix sexo
    combined['sexo'] = combined['sexo'].fillna("V")
    
    #fix ind_empleado
    combined['ind_empleado'] = combined['ind_empleado'].fillna("N")
    
    #fix segmento
    combined["segmento"] = combined["segmento"].fillna("02 - PARTICULARES")
    
    # fixing age
    combined['age'] = pd.to_numeric(combined['age'], errors='coerce')

    combined.loc[combined.age < 18, "age"] = \
        combined.loc[(combined.age > 18) &
                     (combined.age <= 30), "age"].mean(skipna=True)
    combined.loc[combined.age > 100, "age"] = \
        combined.loc[(combined.age > 30) &
                     (combined.age <= 100), "age"].mean(skipna=True)
    combined['age'].fillna(combined['age'].mean(), inplace=True)
    combined['age'] = combined['age'].astype(int)

    # fix ind_nuevo
    combined.loc[combined.ind_nuevo.isnull(), 'ind_nuevo'] = 1

    # fix antiguedad
    combined['antiguedad'] = pd.to_numeric(combined['antiguedad'],
                                           errors='coerce')
    combined.loc[combined.antiguedad.isnull(), 'antiguedad'] = \
        combined.antiguedad.min()
    combined.loc[combined.antiguedad < 0, 'antiguedad'] = 0

    # fix indrel
    combined.loc[combined.indrel.isnull(), 'indrel'] = 1

    # drop useless cols
    combined.drop(['tipodom', 'cod_prov'], axis=1, inplace=True)

    # fix ind_actividad_cliente
    combined['ind_actividad_cliente'] = pd.to_numeric(combined['ind_actividad_cliente'],
                                           errors='coerce')
    combined.loc[combined.ind_actividad_cliente.isnull(),
                 "ind_actividad_cliente"] = \
        combined.ind_actividad_cliente.median()

    # fix city name
    combined.loc[combined.nomprov ==
                 "CORU\xc3\x91A, A", "nomprov"] = "CORUNA, A"
    combined.loc[combined.nomprov.isnull(), 'nomprov'] = 'UNKNOWN'

    # fix incomes
    combined['renta'] = pd.to_numeric(combined['renta'], errors='coerce')

    grouped = combined.groupby('nomprov').\
        agg({'renta': lambda x: x.median(skipna=True)}).reset_index()
    new_incomes = pd.merge(combined, grouped,
                           how='inner',
                           on='nomprov').loc[:, ['nomprov', 'renta_y']]

    new_incomes = new_incomes.\
        rename(columns={"renta_y": "renta"}).\
        sort_values("renta").sort_values("nomprov")

    combined.sort_values("nomprov", inplace=True)
    combined = combined.reset_index()
    new_incomes = new_incomes.reset_index()
    combined.loc[combined.renta.isnull(), "renta"] = \
        new_incomes.loc[combined.renta.isnull(), "renta"].median()
    combined.sort_values(by='fecha_dato', inplace=True)

    # rest of the columns
    string_data = combined.select_dtypes(include=["object"])
    missing_columns = [col for col in string_data
                       if string_data[col].isnull().any()]
    del string_data

    combined.loc[combined.indfall.isnull(), 'indfall'] = 'N'
    combined.loc[combined.tiprel_1mes.isnull(), 'tiprel_1mes'] = 'A'
    combined.tiprel_1mes = combined.tiprel_1mes.astype('category')

    map_dict = {
        '1.0': '1',
        '1': '1',
        '3.0': '3',
        'P': 'P',
        3.0: '3',
        2.0: '2',
        '3': '3',
        '2.0': '2',
        '4.0': '4',
        '4': '4',
        '2': '2',
        1.0: '1',
        4.0: '4'
    }

    combined.indrel_1mes.fillna('P', inplace=True)
    combined.indrel_1mes = combined.indrel_1mes.apply(lambda x: map_dict[x])
    combined.indrel_1mes = combined.indrel_1mes.astype('category')

    unknown_cols = [col for col in missing_columns if
                    col not in ['indfall', 'tiprel_1mes', 'indrel_1mes']]
    for col in unknown_cols:
        combined.loc[combined[col].isnull(), col] = "UNKNOWN"

    # feature cols
    feature_cols = combined.iloc[:1, ].filter(regex="ind_+.*ult.*").\
        columns.values
    for col in feature_cols:
        combined.loc[combined[col].isnull(), col] = 0
        combined[col] = combined[col].astype(int, errors='ignore')

    del combined['ult_fec_cli_1t'], combined['fecha_alta']
    encoder_cols = ['sexo', 'indrel_1mes', 'pais_residencia', 'ind_empleado',
                'segmento', 'tiprel_1mes', 'indresi', 'indext', 'conyuemp',
                'canal_entrada', 'indfall', 'nomprov']
    if encoders == None:
        encoders = []
        for col in encoder_cols:
            temp_enc = LabelEncoder()
            temp_enc.fit(combined[col])
            combined[col] = temp_enc.transform(combined[col])
            encoders.append(temp_enc)
    else:
        for i in range(len(encoders)):
            combined[encoder_cols[i]] = encoders[i].transform(combined[encoder_cols[i]])
        
    return combined, encoders

In [ ]:
combined, encoders = process_data_from_original_dataframe(combined)

In [ ]:
train_old_4 = combined.loc[combined.fecha_dato == '2015-01-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
train_old_3 = combined.loc[combined.fecha_dato == '2015-02-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
train_old_2 = combined.loc[combined.fecha_dato == '2015-03-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
train_old_1 = combined.loc[combined.fecha_dato == '2015-04-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
train_new   = combined.loc[combined.fecha_dato == '2015-05-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)

test_old_4 = combined.loc[combined.fecha_dato == '2016-01-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
test_old_3 = combined.loc[combined.fecha_dato == '2016-02-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
test_old_2 = combined.loc[combined.fecha_dato == '2016-03-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
test_old_1 = combined.loc[combined.fecha_dato == '2016-04-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
test_new   = combined.loc[combined.fecha_dato == '2016-05-28', :].reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)
del combined

In [ ]:
for df in [train_old_4, train_old_3, train_old_2, train_old_1, train_new, test_old_4, test_old_3, test_old_2, test_old_1, test_new]:
    print(df.shape)

In [ ]:
train_old_4.set_index('ncodpers', inplace=True)
train_old_3.set_index('ncodpers', inplace=True)
train_old_2.set_index('ncodpers', inplace=True)
train_old_1.set_index('ncodpers', inplace=True)
train_new.set_index('ncodpers', inplace=True)

test_old_4.set_index('ncodpers', inplace=True)
test_old_3.set_index('ncodpers', inplace=True)
test_old_2.set_index('ncodpers', inplace=True)
test_old_1.set_index('ncodpers', inplace=True)
test_new.set_index('ncodpers', inplace=True)

In [ ]:
added_products.set_index('ncodpers', inplace=True)
print(added_products.shape)

In [ ]:
train = pd.DataFrame()
train['ncodpers'] = added_products.index
train.set_index('ncodpers', inplace=True)

test = pd.DataFrame()
test['ncodpers'] = test_new.index
test.set_index('ncodpers', inplace=True)

not_found_axis = [index for index in test.index if index not in test_old_1.index]
test = test.drop(not_found_axis, axis=0)

for col in train_new.columns[:18]:
    train['current_' + col] = train_new.loc[added_products.index, col]
    test['current_' + col]  = test_new.loc[test_new.index, col]
    
for col in products:
    train['current_' + col] = train_old_1.loc[added_products.index, col]
    test['current_' + col]  = test_old_1.loc[test.index, col]

In [ ]:
def get_file_string_from_train_data(train_data):
    date_time = datetime.datetime(train_data[0], train_data[1], 28)
    return '_'.join(str(date_time.date()).split('-'))

def get_date_string_from_train_data(train_data):
    date_time = datetime.datetime(train_data[0], train_data[1], 28)
    return '-'.join(str(date_time.date()).split('-'))

def subtract_one_month(train_data):
    if train_data[1] == 1:
        return 2015, 12
    else:
        return train_data[0], train_data[1] - 1
    
som = subtract_one_month

In [ ]:
def generate_train_data(train_data, encoders, train, added_products):
    pass

In [ ]:
def generate_train_data_from_train_data(train_data, encoders, train, added_products):
    print(train_data)
    TRAIN_DATA = train_data

    TRAIN_MONTH_OLD_4 = get_file_string_from_train_data(som(som(som(som(TRAIN_DATA)))))
    TRAIN_MONTH_OLD_3 = get_file_string_from_train_data(som(som(som(TRAIN_DATA))))
    TRAIN_MONTH_OLD_2 = get_file_string_from_train_data(som(som(TRAIN_DATA)))
    TRAIN_MONTH_OLD_1 = get_file_string_from_train_data(som(TRAIN_DATA))
    TRAIN_MONTH_NEW = get_file_string_from_train_data(TRAIN_DATA)

    TRAIN_FILE_OLD_4 = 'data/train_' + TRAIN_MONTH_OLD_4 + '.csv'
    TRAIN_FILE_OLD_3 = 'data/train_' + TRAIN_MONTH_OLD_3 + '.csv'
    TRAIN_FILE_OLD_2 = 'data/train_' + TRAIN_MONTH_OLD_2 + '.csv'
    TRAIN_FILE_OLD_1 = 'data/train_' + TRAIN_MONTH_OLD_1 + '.csv'
    TRAIN_FILE_NEW = 'data/train_' + TRAIN_MONTH_NEW + '.csv'

    ADDED_PRODUCTS_FILE = 'data/added_product_' + TRAIN_MONTH_OLD_1 + '.csv'

    train_old_4 = pd.read_csv(TRAIN_FILE_OLD_4)
    train_old_3 = pd.read_csv(TRAIN_FILE_OLD_3)
    train_old_2 = pd.read_csv(TRAIN_FILE_OLD_2)
    train_old_1 = pd.read_csv(TRAIN_FILE_OLD_1)
    train_new = pd.read_csv(TRAIN_FILE_NEW) #, header=None, names=HEADER

    added_products_temp = pd.read_csv(ADDED_PRODUCTS_FILE)
    
    combined = pd.concat((train_old_4, train_old_3, train_old_2, train_old_1,train_new)).reset_index(drop=True)
    del train_old_4, train_old_3, train_old_2, train_old_1, train_new
    gc.collect()

    combined, encoders = process_data_from_original_dataframe(combined, encoders)
    
    train_old_4 = combined.loc[combined.fecha_dato == '-'.join(TRAIN_MONTH_OLD_4.split('_')), :].reset_index(drop=True)
    train_old_3 = combined.loc[combined.fecha_dato == '-'.join(TRAIN_MONTH_OLD_3.split('_')), :].reset_index(drop=True)
    train_old_2 = combined.loc[combined.fecha_dato == '-'.join(TRAIN_MONTH_OLD_2.split('_')), :].reset_index(drop=True)
    train_old_1 = combined.loc[combined.fecha_dato == '-'.join(TRAIN_MONTH_OLD_1.split('_')), :].reset_index(drop=True)
    train_new   = combined.loc[combined.fecha_dato == '-'.join(TRAIN_MONTH_NEW.split('_')), :].reset_index(drop=True)
    
    del combined
    del train_old_1['index'], train_old_2['index'], train_old_3['index'], train_old_4['index'], train_new['index']
    del train_old_1['fecha_dato'], train_old_2['fecha_dato'], train_old_3['fecha_dato'], train_old_4['fecha_dato'], train_new['fecha_dato']
    gc.collect()
    
    train_old_4.set_index('ncodpers', inplace=True)
    train_old_3.set_index('ncodpers', inplace=True)
    train_old_2.set_index('ncodpers', inplace=True)
    train_old_1.set_index('ncodpers', inplace=True)
    train_new.set_index('ncodpers', inplace=True)

    added_products_temp.set_index('ncodpers', inplace=True)
    
    train_temp = pd.DataFrame()
    train_temp['ncodpers'] = added_products_temp.index
    train_temp.set_index('ncodpers', inplace=True)

    for col in train_new.columns[:18]:
        train_temp['current_' + col] = train_new.loc[added_products_temp.index, col]

    for col in products:
        train_temp['current_' + col] = train_old_1.loc[added_products_temp.index, col]
    
    del train_old_1, train_old_2, train_old_3, train_old_4
    gc.collect()
    train_temp['month'] = [TRAIN_DATA[1]] * train_temp.shape[0]
    
    print(train.shape, train_temp.shape, added_products.shape, added_products_temp.shape)
    train = pd.concat((train, train_temp))
    added_products = pd.concat((added_products, added_products_temp))
    
    print(train.shape, train_temp.shape, added_products.shape, added_products_temp.shape)
    return encoders, train, added_products

In [ ]:
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 6), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 7), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 8), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 9), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 10), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 11), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2015, 12), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2016, 1), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2016, 2), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2016, 3), encoders, train, added_products)
print(f"---------------{train.shape}---------------")
encoders, train, added_products = generate_train_data_from_train_data((2016, 4), encoders, train, added_products)
print(f"---------------{train.shape}---------------")

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(added_products.added_product)
added_products['encoded_products'] = label_encoder.\
    transform(added_products['added_product'])

In [ ]:
train

In [ ]:
print(train.shape)
print(added_products.shape)

In [ ]:
xgb_main = xgb.DMatrix(train, label=added_products.encoded_products)

def map7eval(preds, dtrain):
    actual = dtrain.get_label()
    predicted = preds.argsort(axis=1)[:,-np.arange(1,8)]
    metric = 0.
    for i in range(7):
        metric += np.sum(actual==predicted[:,i])/(i+1)
    metric /= actual.shape[0]
    return 'MAP@7', metric

param = {}
param['objective'] = 'multi:softprob'
param['eta'] = 0.05
param['max_depth'] = 6
param['silent'] = 0
param['num_class'] = len(label_encoder.classes_)
param['eval_metric'] = "mlogloss"
param['min_child_weight'] = 2
param['subsample'] = 0.9
param['colsample_bytree'] = 0.9
param['seed'] = 1428
num_rounds = 400
plist = list(param.items())

evals_result = {}
evallist = [(xgb_main, 'train')]

In [ ]:
xgb_model = xgb.train(plist, xgb_main, num_rounds, evallist, feval=map7eval, evals_result=evals_result)

In [ ]:
xgb.plot_importance(xgb_model)

In [ ]:
test['month'] = [6] * test.shape[0]

final_test = xgb.DMatrix(test)
preds = xgb_model.predict(final_test)

top_t_products = [label_encoder.inverse_transform(np.argsort(pred, axis=0)[::-1][:7]) for pred in preds]

test['xgb_preds'] = [' '.join(x) for x in top_t_products]
test['added_products'] = ['ind_recibo_ult1'] * test.shape[0]
for i in test.index:
    zipped = zip(products,
                 test.loc[i, 'current_ind_ahor_fin_ult1':'current_ind_recibo_ult1'])
    products = [x[0] for x in
                [y for y in zipped if y[1] == 1]]
    pred_products = test.loc[i, 'xgb_preds'].split()
    prod_string = ' '.join(filter(lambda x: x not in products, pred_products))
    test.at[i, 'added_products'] = prod_string

In [ ]:
import seaborn as sns

p2 = pd.DataFrame()
p2['index'] = range(0, num_rounds)
p2['map7_eval_score'] = evals_result['train']['MAP@7']

plt.rcParams['figure.figsize'] = (12, 10)
sns.regplot(x='index', y='map7_eval_score', data = p2)
plt.show()

In [ ]:
import pickle

pickle_description = '11. Contains test_ver2.ncodpers, and its respective preds, and label_encoder'
filename = "model_xgb"
test_ncodpers = list(test.index)
dump_object = (pickle_description, test_ncodpers, preds, label_encoder)

pickle.dump(dump_object, open(filename+'.pickle', 'wb'))

In [ ]:
test

In [ ]:
test[["xgb_preds"]].to_csv("preds.csv")